# Tutorial 3: Model Grids and Performance

This tutorial explores pre-computed model grids for fast SED generation and performance optimization.

## Topics Covered

1. **GridGenerator** for creating custom grids
2. **StarGrid** for fast SED lookups
3. **Performance comparison**: StarGrid vs StarEvolTrack
4. **Grid resolution** and accuracy trade-offs
5. **Loading and inspecting existing grids**
6. **Reddening coefficients** in grids

## Prerequisites

This tutorial requires the following brutus data files:
- `MIST_1.2_EEPtrk.h5` - MIST evolutionary tracks
- `nn_c3k.h5` - Neural network for bolometric corrections
- `grid_mist_v9.h5` (optional) - Pre-computed MIST grid
- `grid_bayestar_v5.h5` (optional) - Pre-computed Bayestar grid

If you don't have these files, run the optional download cell below.

In [ ]:
# Optional: Download required data files (only needed if not already cached)
# This tutorial requires MIST evolutionary tracks, the C3K neural network,
# and optionally pre-computed model grids.
# Uncomment the lines below to download them.

# from brutus.data import fetch_tracks, fetch_nns, fetch_grids
# fetch_tracks()     # ~60 MB  -- MIST evolutionary tracks
# fetch_nns()        # ~50 MB  -- Neural network for bolometric corrections
# fetch_grids()      # ~600 MB -- Pre-computed MIST grid (optional)

In [ ]:
# Imports and setup
import time
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from tutorial_utils import (
    setup_tutorial,
    find_brutus_data_file,
    save_figure as _save_fig,
    print_section,
)

info = setup_tutorial(3, title="Tutorial 03: Model Grids and Performance")
plots_dir = info['plot_dir']


def save_figure(fig, name):
    """Save figure to this tutorial's plot directory."""
    _save_fig(fig, 3, name)

## Section 1: Creating Model Grids with GridGenerator

GridGenerator creates pre-computed model grids with reddening coefficients.
This enables fast SED generation for large-scale fitting applications.

### Key Concepts

- **Grid parameters**: Define the parameter space to sample (mass, metallicity, EEP, etc.)
- **Reddening coefficients**: Store polynomial coefficients for fast extinction calculations
- **Reference distance**: All grids use 1 kpc (1000 pc) reference distance
- **Storage format**: HDF5 files with labels, parameters, and magnitude coefficients

In [ ]:
from brutus.core import EEPTracks, GridGenerator
from brutus.data import filters

# Initialize components
print("Loading MIST tracks and neural networks...")
mistfile = find_brutus_data_file('MIST_1.2_EEPtrk.h5')
nnfile = find_brutus_data_file('nn_c3k.h5')

# Use subset of filters for speed
filt = filters.ps[:3] + filters.tmass[:2]  # g,r,i + J,H
print(f"Using filters: {', '.join(filt)}")

tracks = EEPTracks(mistfile=mistfile, verbose=False)
print("Loaded EEPTracks")

In [ ]:
# Create grid generator
print("Initializing GridGenerator...")
generator = GridGenerator(
    tracks=tracks,
    nnfile=nnfile,
    filters=filt,
    verbose=True
)

print("\nDefining grid parameters for a small demo grid...")

# Define small grid for demonstration
mini_grid = np.array([0.5, 0.8, 1.0, 1.5, 2.0])  # 5 masses
feh_grid = np.array([-1.0, 0.0, 0.3])  # 3 metallicities
eep_grid = np.linspace(300, 500, 21)  # 21 EEP points (MS only)
afe_grid = np.array([0.0])  # Single alpha
smf_grid = np.array([0.0])  # No binaries for speed

total_models = len(mini_grid) * len(feh_grid) * len(eep_grid) * len(afe_grid) * len(smf_grid)
print(f"\nGrid dimensions:")
print(f"  Masses: {len(mini_grid)} points")
print(f"  Metallicities: {len(feh_grid)} points")
print(f"  EEPs: {len(eep_grid)} points")
print(f"  Total models to generate: {total_models}")

In [ ]:
# Generate the grid
print("\nGenerating model grid (this may take a minute)...")
start_time = time.time()

generator.make_grid(
    mini_grid=mini_grid,
    feh_grid=feh_grid,
    eep_grid=eep_grid,
    afe_grid=afe_grid,
    smf_grid=smf_grid,
    av_grid=np.linspace(0, 3, 4),  # For reddening coefficients
    rv_grid=np.array([3.1]),
    verbose=False
)

gen_time = time.time() - start_time
print(f"\nGrid generation complete in {gen_time:.1f} seconds")
print(f"  Models generated: {len(generator.grid_labels)}")
print(f"  Valid models: {generator.grid_sel.sum()}")
print(f"  Time per model: {gen_time/generator.grid_sel.sum()*1000:.1f} ms")

In [ ]:
# Save the demo grid
demo_grid_file = plots_dir / 'demo_grid.h5'
print(f"\nSaving demo grid to {demo_grid_file}")

with h5py.File(demo_grid_file, 'w') as f:
    # Save valid models only
    valid = generator.grid_sel
    f.create_dataset('labels', data=generator.grid_labels[valid])
    f.create_dataset('parameters', data=generator.grid_params[valid])
    f.create_dataset('mag_coeffs', data=generator.grid_seds[valid])
    
    # Add metadata
    f.attrs['filters'] = str(filt)
    f.attrs['reference_distance_pc'] = 1000.0
    f.attrs['rv'] = 3.1
    f.attrs['generation_time'] = gen_time
    f.attrs['n_models'] = valid.sum()

print(f"Saved {valid.sum()} models to HDF5 file")
print(f"  File size: {demo_grid_file.stat().st_size / 1024**2:.2f} MB")

## Section 2: StarGrid for Fast SED Lookups

StarGrid wraps a pre-computed grid of magnitude coefficients for interactive
queries.  Given stellar parameters (mass, EEP, metallicity), it interpolates
the grid to return an SED at any extinction and distance.

### Role in the brutus pipeline

- **Interactive exploration**: `StarGrid.get_seds()` lets you query
  individual models by physical parameters via multi-linear interpolation.
- **BruteForce fitting**: The fitting engine accesses the raw `mag_coeffs`
  array directly, evaluating *every* grid model against each observed star
  in a single Numba-compiled pass — no per-model neural network call needed.
- **Compact storage**: Each model stores only 3 numbers per band
  (`mag`, `R`, `dR/dRv`), so reddening at any `(A_V, R_V)` is a
  simple linear operation.

In [ ]:
from brutus.core import StarGrid
from brutus.data import load_models

# Try to load a pre-computed grid
print("Loading pre-computed MIST grid...")

try:
    grid_file = find_brutus_data_file('grid_mist_v9.h5')
    filt_grid = filters.ps[:5] + filters.tmass  # PS + 2MASS
    
    models, labels, mask = load_models(grid_file, filters=filt_grid)
    print(f"Loaded {len(models):,} models")
    print(f"  Filters: {', '.join(filt_grid)}")
    
    # Initialize StarGrid (positional: models, models_labels)
    grid = StarGrid(models, labels, filters=filt_grid)
    print("StarGrid initialized")
    
    grid_available = True
    
except FileNotFoundError:
    print("Pre-computed grid not found")
    print("  Using demo grid from Section 1 instead...")
    
    # Load the demo grid we just created
    with h5py.File(demo_grid_file, 'r') as f:
        labels = f['labels'][:]
        models = f['mag_coeffs'][:]
    
    grid = StarGrid(models, labels, filters=filt)
    grid_available = False

In [ ]:
# Test StarGrid performance
print("\nTesting StarGrid performance...")

# Generate 100 SEDs for timing
n_test = 100
start = time.time()

for _ in range(n_test):
    try:
        mag, params, _ = grid.get_seds(
            mini=1.0, feh=0.0, eep=400,
            av=0.5, rv=3.1, dist=1000.0
        )
    except:
        # May fail if parameters are out of grid range
        pass

grid_time = (time.time() - start) / n_test
print(f"  Average time per SED: {grid_time*1000:.2f} ms")

# Show example output
mag, params, _ = grid.get_seds(
    mini=1.0, feh=0.0, eep=400,
    av=0.0, rv=3.1, dist=1000.0
)

print(f"\nExample SED for 1 M_sun star at 1 kpc:")
for i, f in enumerate(grid.filters[:min(5, len(grid.filters))]):
    print(f"  {f}: {mag[i]:.2f} mag")

if isinstance(params, dict) and 'loga' in params:
    print(f"\nDerived parameters:")
    print(f"  Age: {10**(params['loga'])/1e9:.2f} Gyr")
    print(f"  log L/L_sun: {params.get('logl', 'N/A'):.2f}")
    print(f"  log Teff: {params.get('logt', 'N/A'):.2f}")

## Section 3: Performance Comparison

The real performance advantage of pre-computed grids shows up in
batch evaluation.  During fitting, `BruteForce` applies reddening
to **all** grid models at once using the stored magnitude coefficients.
Each evaluation is a simple linear operation — 3 multiplies and 2 adds
per model per band, compiled with Numba — instead of an expensive
neural-network forward pass.

Below we compare:
1. **StarEvolTrack**: N individual on-the-fly SED evaluations (track interpolation + neural network per call)
2. **`get_seds` on pre-computed coefficients**: Applying reddening to N stored `mag_coeffs` in a single batch call

In [ ]:
from brutus.core import StarEvolTrack, get_seds

# Setup
filt_test = filters.ps[:3] + filters.tmass[:2]
tracks = EEPTracks(mistfile=mistfile, verbose=False)
star_evol = StarEvolTrack(tracks=tracks, nnfile=nnfile, filters=filt_test, verbose=False)

# ------------------------------------------------------------------
# Compare: N individual StarEvolTrack calls  vs  one batch get_seds call
# This mirrors what BruteForce does — it never calls StarGrid.get_seds()
# for single-point lookups. Instead it passes the entire mag_coeffs array
# to _get_seds() in one Numba-compiled pass.
# ------------------------------------------------------------------
rng = np.random.default_rng(42)
test_minis = rng.uniform(0.5, 2.0, 2000)
test_fehs = rng.uniform(-1, 0.3, 2000)
test_eeps = rng.uniform(300, 500, 2000)

# Warm up Numba JIT (first call compiles the function)
if grid_available:
    _ = get_seds(models[:2], av=np.array([0.5, 1.0]), rv=np.array([3.1, 3.1]))

n_values = [10, 50, 100, 500, 1000]
evol_times = []
grid_times = []

print("Per-model cost: StarEvolTrack (NN) vs get_seds (pre-computed coefficients)\n")

for n in n_values:
    # Method 1: N individual StarEvolTrack.get_seds() calls
    #   Each call runs: track interpolation -> NN forward pass -> extinction
    start = time.time()
    for i in range(n):
        try:
            star_evol.get_seds(mini=test_minis[i], feh=test_fehs[i],
                               eep=test_eeps[i], dist=1000.0)
        except Exception:
            pass
    t_evol = time.time() - start
    evol_times.append(t_evol)

    # Method 2: Single get_seds() call on N pre-computed mag_coeffs
    #   Just applies: m = mag + Av * (R + Rv * dR/dRv) for all N models
    if grid_available:
        idx = rng.choice(len(models), n, replace=False)
        batch_coeffs = models[idx]  # (N, Nfilt, 3)
        av_arr = rng.uniform(0, 2, n)
        rv_arr = np.full(n, 3.1)

        start = time.time()
        _ = get_seds(batch_coeffs, av=av_arr, rv=rv_arr)
        t_grid = time.time() - start
        grid_times.append(t_grid)

        speedup = t_evol / t_grid if t_grid > 0 else float('inf')
        print(f"  N={n:5d}:  StarEvolTrack = {t_evol*1000:8.1f} ms,  "
              f"get_seds = {t_grid*1000:8.3f} ms  "
              f"(batch {speedup:.0f}x faster)")
    else:
        grid_times.append(None)
        print(f"  N={n:5d}:  StarEvolTrack = {t_evol*1000:8.1f} ms")

print("\nStarEvolTrack cost is dominated by the neural-network forward pass")
print("and Python-level track interpolation on each call.")
print("get_seds applies a simple linear formula to pre-computed coefficients,")
print("so the batch cost grows negligibly with N.")

In [ ]:
# Performance comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

n_arr = np.array(n_values)
evol_arr = np.array(evol_times) * 1000  # ms

# Panel 1: Total time vs number of models
ax = axes[0]
ax.loglog(n_arr, evol_arr, 'bo-', lw=2, label='StarEvolTrack (NN)')

if grid_available and all(g is not None for g in grid_times):
    grid_arr = np.array(grid_times) * 1000
    ax.loglog(n_arr, grid_arr, 'rs-', lw=2, label='get_seds (pre-computed)')

ax.set_xlabel('Number of Models')
ax.set_ylabel('Total Time (ms)')
ax.set_title('Total Evaluation Time')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Time per model (amortized)
ax = axes[1]
ax.loglog(n_arr, evol_arr / n_arr, 'bo-', lw=2, label='StarEvolTrack')

if grid_available and all(g is not None for g in grid_times):
    ax.loglog(n_arr, grid_arr / n_arr, 'rs-', lw=2, label='get_seds')

ax.set_xlabel('Number of Models')
ax.set_ylabel('Time per Model (ms)')
ax.set_title('Amortized Cost per Model')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Speedup factor
ax = axes[2]
if grid_available and all(g is not None for g in grid_times):
    speedups = evol_arr / grid_arr
    ax.semilogx(n_arr, speedups, 'go-', lw=2, markersize=8)
    ax.set_xlabel('Number of Models')
    ax.set_ylabel('Speedup Factor')
    ax.set_title('Batch Speedup (StarEvolTrack / get_seds)')
    ax.grid(True, alpha=0.3)
    ax.axhline(1, color='gray', ls='--', lw=0.8)
else:
    ax.text(0.5, 0.5, 'Pre-computed grid\nnot available',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('Batch Speedup')

plt.suptitle('Pre-computed Grid Performance Advantage', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'performance_comparison')
plt.show()

print("\nPre-computed grids trade memory for speed: store mag_coeffs once,")
print("then evaluate any (Av, Rv, dist) combination with trivial arithmetic.")

## Section 4: Grid Resolution and Accuracy Trade-offs

Grid resolution affects interpolation accuracy and storage requirements.
Here we measure actual interpolation error by comparing SEDs computed at
a coarse grid spacing (and linearly interpolated) against a dense "truth"
computed at every integer EEP.

### Key Trade-offs

- **Finer grids**: Better accuracy but larger storage and longer generation time
- **Coarser grids**: Faster to generate and smaller on disk, but higher interpolation error
- **Non-uniform spacing**: Dense sampling in rapidly-varying regions (e.g. the TAMS) can help

In [ ]:
# Measure actual interpolation error for different EEP grid spacings
#
# Approach:
#   1. Compute g-r color at every integer EEP (spacing=1) as "truth"
#   2. Subsample at coarser spacings and linearly interpolate back
#   3. Compare interpolated values to truth

print("Computing dense truth (g-r color at every integer EEP)...\n")

eep_dense = np.arange(300, 501, 1)  # 201 EEPs, spacing = 1
colors_dense = []

for eep in eep_dense:
    try:
        mags, _, _ = star_evol.get_seds(mini=1.0, feh=0.0, eep=eep, dist=100.0)
        colors_dense.append(mags[0] - mags[1])  # g-r color
    except Exception:
        colors_dense.append(np.nan)

colors_dense = np.array(colors_dense)
valid_dense = np.isfinite(colors_dense)
print(f"  Dense truth: {np.sum(valid_dense)}/{len(eep_dense)} valid EEPs")

# Test different grid spacings
spacings = [5, 10, 20, 50]
resolution_results = {}

for spacing in spacings:
    eep_coarse = np.arange(300, 501, spacing)
    colors_coarse = []

    for eep in eep_coarse:
        try:
            mags, _, _ = star_evol.get_seds(mini=1.0, feh=0.0, eep=eep, dist=100.0)
            colors_coarse.append(mags[0] - mags[1])
        except Exception:
            colors_coarse.append(np.nan)

    colors_coarse = np.array(colors_coarse)
    valid_coarse = np.isfinite(colors_coarse)

    if np.sum(valid_coarse) > 1:
        # Interpolate coarse grid to dense EEP locations
        interp_colors = np.interp(
            eep_dense,
            eep_coarse[valid_coarse],
            colors_coarse[valid_coarse]
        )

        # Residuals where both dense truth and interpolation are valid
        both_valid = valid_dense & np.isfinite(interp_colors)
        residuals = np.full_like(colors_dense, np.nan)
        residuals[both_valid] = (interp_colors[both_valid] - colors_dense[both_valid]) * 1000  # mmag

        rms = np.nanstd(residuals) if np.any(both_valid) else np.nan
        max_err = np.nanmax(np.abs(residuals)) if np.any(both_valid) else np.nan

        resolution_results[spacing] = {
            'eep_coarse': eep_coarse,
            'colors_coarse': colors_coarse,
            'valid_coarse': valid_coarse,
            'interp_colors': interp_colors,
            'residuals': residuals,
            'rms': rms,
            'max_err': max_err,
        }

        print(f"  EEP spacing = {spacing:2d}: {np.sum(valid_coarse):3d} grid points,  "
              f"RMS = {rms:.2f} mmag,  max = {max_err:.2f} mmag")

In [ ]:
# Grid resolution visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel 1: g-r color vs EEP — truth and coarse-grid interpolations
ax = axes[0, 0]
ax.plot(eep_dense[valid_dense], colors_dense[valid_dense],
        'k-', lw=1.5, label='Truth (dEEP=1)', zorder=5)

cmap_res = plt.cm.Set1
for i, spacing in enumerate(spacings):
    if spacing not in resolution_results:
        continue
    r = resolution_results[spacing]
    vc = r['valid_coarse']
    ax.plot(r['eep_coarse'][vc], r['colors_coarse'][vc],
            'o', color=cmap_res(i / len(spacings)), markersize=4,
            label=f'dEEP={spacing}', alpha=0.8)

ax.set_xlabel('EEP')
ax.set_ylabel('g - r  (mag)')
ax.set_title('Color vs EEP at Different Resolutions')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: Interpolation residuals vs EEP
ax = axes[0, 1]
for i, spacing in enumerate(spacings):
    if spacing not in resolution_results:
        continue
    r = resolution_results[spacing]
    valid = np.isfinite(r['residuals'])
    if np.any(valid):
        ax.plot(eep_dense[valid], r['residuals'][valid],
                '-', color=cmap_res(i / len(spacings)), lw=1,
                label=f'dEEP={spacing} (RMS={r["rms"]:.1f} mmag)', alpha=0.8)

ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('EEP')
ax.set_ylabel('Interpolation Error (mmag)')
ax.set_title('Actual Interpolation Residuals')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: RMS error vs grid spacing
ax = axes[1, 0]
spacings_arr = []
rms_arr = []
max_arr = []

for spacing in spacings:
    if spacing in resolution_results:
        spacings_arr.append(spacing)
        rms_arr.append(resolution_results[spacing]['rms'])
        max_arr.append(resolution_results[spacing]['max_err'])

spacings_arr = np.array(spacings_arr)
rms_arr = np.array(rms_arr)
max_arr = np.array(max_arr)

ax.semilogy(spacings_arr, rms_arr, 'ko-', lw=2, markersize=8, label='RMS error')
ax.semilogy(spacings_arr, max_arr, 'r^--', lw=1.5, markersize=8, label='Max error')
ax.set_xlabel('EEP Grid Spacing')
ax.set_ylabel('Interpolation Error (mmag)')
ax.set_title('Accuracy vs Resolution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 4: Storage requirements from actual grid files
ax = axes[1, 1]

grid_info = []
for filename, name in [('grid_mist_v9.h5', 'MIST v9'),
                        ('grid_bayestar_v5.h5', 'Bayestar v5')]:
    try:
        filepath = Path(find_brutus_data_file(filename))
        size_mb = filepath.stat().st_size / 1024**2
        with h5py.File(filepath, 'r') as f:
            n_models = len(f['labels'])
        grid_info.append((name, n_models, size_mb))
    except FileNotFoundError:
        pass

# Add the demo grid
grid_info.append(('Demo grid', int(generator.grid_sel.sum()),
                  demo_grid_file.stat().st_size / 1024**2))

names = [g[0] for g in grid_info]
sizes = [g[2] for g in grid_info]
n_models_list = [g[1] for g in grid_info]

bars = ax.bar(names, sizes, color=['steelblue', 'coral', 'gray'][:len(names)], alpha=0.7)
ax.set_ylabel('File Size (MB)')
ax.set_title('Pre-computed Grid Storage')
ax.grid(True, alpha=0.3, axis='y')

for bar, nm in zip(bars, n_models_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{nm:,}\nmodels', ha='center', va='bottom', fontsize=8)

plt.suptitle('Grid Resolution and Storage Trade-offs', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'grid_resolution')
plt.show()

print("\nGrid resolution analysis complete")

## Section 5: Working with Existing Grids

Brutus provides pre-computed grids for common use cases.
Let's explore the available grids and their properties.

### Available Grids

- **grid_mist_v9.h5**: Full MIST stellar evolution models
- **grid_bayestar_v5.h5**: Empirical Pan-STARRS models for dust mapping

In [ ]:
# Check for available grids
grids_to_check = [
    ('grid_mist_v9.h5', 'MIST v9', 'Full MIST stellar evolution models'),
    ('grid_bayestar_v5.h5', 'Bayestar v5', 'Empirical Pan-STARRS models'),
]

loaded_grids = []

for (filename, name, description) in grids_to_check:
    print(f"\n{name}: {description}")
    print("-" * 50)
    
    try:
        filepath = Path(find_brutus_data_file(filename))
        
        # Open to inspect
        with h5py.File(filepath, 'r') as f:
            print(f"  File: {filename}")
            print(f"  Size: {filepath.stat().st_size / 1024**2:.1f} MB")
            print(f"  Keys: {list(f.keys())}")
            
            if 'labels' in f:
                grid_labels = f['labels'][:]
                print(f"  Number of models: {len(grid_labels):,}")
                print(f"  Label fields: {grid_labels.dtype.names}")
                
                # Get parameter ranges
                if 'mini' in grid_labels.dtype.names:
                    print(f"  Mass range: {grid_labels['mini'].min():.2f} - {grid_labels['mini'].max():.2f} M_sun")
                if 'feh' in grid_labels.dtype.names:
                    print(f"  [Fe/H] range: {grid_labels['feh'].min():.2f} - {grid_labels['feh'].max():.2f}")
                if 'Mr' in grid_labels.dtype.names:
                    print(f"  M_r range: {grid_labels['Mr'].min():.1f} - {grid_labels['Mr'].max():.1f}")
                
            if 'mag_coeffs' in f:
                mags = f['mag_coeffs']
                # mag_coeffs can be a structured array (filter names as fields,
                # each with 3 coefficients) or a regular (N, Nfilt, 3) array.
                if mags.dtype.names:
                    n_filters = len(mags.dtype.names)
                    print(f"  Magnitude coefficients: {len(mags):,} models x {n_filters} filters x 3 coefficients")
                    print(f"  Filter names: {', '.join(mags.dtype.names[:5])}...")
                else:
                    print(f"  Magnitude array shape: {mags.shape}")
            
            # Check metadata
            if f.attrs:
                print("  Metadata:")
                for key in list(f.attrs.keys())[:5]:
                    print(f"    {key}: {f.attrs[key]}")
            
            loaded_grids.append((name, grid_labels, filename))
            
    except FileNotFoundError:
        print(f"  [Not found - optional file]")

In [ ]:
# Visualize grid coverage if available
if loaded_grids:
    fig, axes = plt.subplots(len(loaded_grids), 3, figsize=(15, 5*len(loaded_grids)))
    if len(loaded_grids) == 1:
        axes = axes.reshape(1, -1)

    for idx, (name, labels, filename) in enumerate(loaded_grids):
        # Panel 1: Parameter space coverage (2D histogram to avoid aliasing)
        ax = axes[idx, 0]

        if 'mini' in labels.dtype.names and 'feh' in labels.dtype.names:
            # MIST grid — log-spaced mass bins
            mass_bins = np.logspace(np.log10(labels['mini'].min()),
                                    np.log10(labels['mini'].max()), 80)
            feh_bins = np.linspace(labels['feh'].min(), labels['feh'].max(), 80)
            h = ax.hist2d(labels['feh'], labels['mini'],
                          bins=[feh_bins, mass_bins],
                          cmap='viridis', cmin=1)
            plt.colorbar(h[3], ax=ax, label='Count')
            ax.set_xlabel('[Fe/H]')
            ax.set_ylabel('Initial Mass (M_sun)')
            ax.set_yscale('log')
        elif 'Mr' in labels.dtype.names:
            # Bayestar grid
            feh_bins = np.linspace(labels['feh'].min(), labels['feh'].max(), 60)
            mr_bins = np.linspace(labels['Mr'].min(), labels['Mr'].max(), 60)
            h = ax.hist2d(labels['feh'], labels['Mr'],
                          bins=[feh_bins, mr_bins],
                          cmap='viridis', cmin=1)
            plt.colorbar(h[3], ax=ax, label='Count')
            ax.set_xlabel('[Fe/H]')
            ax.set_ylabel('M_r')
            ax.invert_yaxis()

        ax.set_title(f'{name} Coverage')

        # Panel 2: Primary parameter distribution
        ax = axes[idx, 1]

        if 'mini' in labels.dtype.names:
            ax.hist(labels['mini'], bins=np.logspace(
                np.log10(labels['mini'].min()),
                np.log10(labels['mini'].max()), 50),
                alpha=0.7, color='steelblue', edgecolor='white', linewidth=0.3)
            ax.set_xlabel('Initial Mass (M_sun)')
            ax.set_xscale('log')
        elif 'Mr' in labels.dtype.names:
            ax.hist(labels['Mr'], bins=50, alpha=0.7, color='coral',
                    edgecolor='white', linewidth=0.3)
            ax.set_xlabel('M_r')

        ax.set_ylabel('Number of Models')
        ax.set_title(f'{name} Distribution')
        ax.grid(True, alpha=0.3)

        # Panel 3: Metallicity distribution
        ax = axes[idx, 2]
        if 'feh' in labels.dtype.names:
            ax.hist(labels['feh'], bins=50, alpha=0.7, color='forestgreen',
                    edgecolor='white', linewidth=0.3)
            ax.set_xlabel('[Fe/H]')
            ax.set_ylabel('Number of Models')
            ax.set_title('Metallicity Distribution')
            ax.grid(True, alpha=0.3)

    plt.suptitle('Pre-computed Model Grids', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'existing_grids')
    plt.show()

    print("\nGrid inspection complete")
else:
    print("\nNo pre-computed grids available for visualization")

## Section 6: Understanding Reddening Coefficients

Model grids store reddening coefficients to enable fast extinction calculations
without needing to re-evaluate the neural network for each value of $A_V$.

### Reddening Parameterization

For each model and filter, the `mag_coeffs` array stores 3 values:
- **`mag`**: The unreddened apparent magnitude at the reference distance (1 kpc)
- **`R`**: The reddening vector component at $R_V = 0$
- **`dR/dR_V`**: The change in the reddening vector per unit $R_V$

The reddened magnitude in each band is then:

$$m_\lambda = \text{mag} + A_V \times (R + R_V \times dR/dR_V)$$

This parameterization captures the full wavelength-dependent extinction law
with only 3 stored numbers per model per band, enabling fast evaluation for
any combination of $A_V$ and $R_V$.

In [ ]:
# Demonstrate the reddening coefficient structure
print("Reddening Coefficient Structure in brutus Grids\n")

print("Each model and filter stores 3 coefficients: [mag, R, dR/dRv]")
print()
print("  mag   : unreddened magnitude at 1 kpc reference distance")
print("  R     : reddening vector component at Rv=0")
print("  dR/dRv: change in reddening vector per unit Rv")
print()
print("Reddened magnitude:")
print("  m_lambda = mag + Av * (R + Rv * dR/dRv)")
print()
print("Benefits of this parameterization:")
print("  - Fast evaluation for any (Av, Rv) combination")
print("  - No neural network re-evaluation needed")
print("  - Linear in Av, linear in Rv")
print("  - Only 3 numbers per model per band")
print()

# Show example from the demo grid
# grid_seds is a structured array: each field is a filter name with shape (3,)
seds = generator.grid_seds
seds_fields = seds.dtype.names
print(f"Example from demo grid:")
print(f"  {len(seds)} models, {len(seds_fields)} filters, 3 coefficients each")

# Pick a valid model
valid_idx = np.where(generator.grid_sel)[0][0]
print(f"\nModel {valid_idx} coefficients:")
for fname in seds_fields:
    coeffs = seds[valid_idx][fname]
    print(f"  {fname}: mag={coeffs[0]:.2f}, R={coeffs[1]:.4f}, dR/dRv={coeffs[2]:.4f}")

In [ ]:
# Visualize reddening coefficients from the demo grid
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

valid_mask = generator.grid_sel
seds = generator.grid_seds[valid_mask]
seds_fields = seds.dtype.names  # Filter names

# Panel 1: Reddening vector (R) per band across all valid models
ax = axes[0]
for fname in seds_fields:
    r_vals = np.array([s[fname][1] for s in seds])  # R coefficient (index 1)
    finite = np.isfinite(r_vals)
    if np.any(finite):
        ax.hist(r_vals[finite], bins=30, alpha=0.5, label=fname)

ax.set_xlabel('R (reddening vector at Rv=0)')
ax.set_ylabel('Number of Models')
ax.set_title('Reddening Vector Distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: dR/dRv per band — shows how Rv changes the extinction
ax = axes[1]
for fname in seds_fields:
    dr_vals = np.array([s[fname][2] for s in seds])  # dR/dRv coefficient (index 2)
    finite = np.isfinite(dr_vals)
    if np.any(finite):
        ax.hist(dr_vals[finite], bins=30, alpha=0.5, label=fname)

ax.set_xlabel('dR/dRv')
ax.set_ylabel('Number of Models')
ax.set_title('Rv Sensitivity per Band')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: Color excess E(B-V) = Av / Rv for different Rv values
ax = axes[2]
av_range = np.linspace(0, 3, 50)

for rv in [2.5, 3.1, 3.3, 4.0, 5.0]:
    ebv = av_range / rv
    ax.plot(av_range, ebv, label=f'R(V) = {rv}', lw=2)

ax.set_xlabel('A(V)')
ax.set_ylabel('E(B-V)')
ax.set_title('Color Excess Relations')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Reddening Coefficients from Demo Grid', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'reddening_coefficients')
plt.show()

print("\nReddening coefficient analysis complete")

## Section: Reddened SED Generation with `get_seds`

The `get_seds` function is the core utility for applying dust reddening to
pre-computed magnitude coefficients. Each model stores a compact representation
of how its photometry changes with extinction:

**`mag_coeffs` array** has shape `(Nmodels, Nbands, 3)` where the 3 coefficients are:
- **`mag`** (index 0): The unreddened apparent magnitude at the reference distance (1 kpc).
- **`R`** (index 1): The reddening vector component at `R(V) = 0`.
- **`dR/dRv`** (index 2): The change in the reddening vector per unit `R(V)`.

The reddened magnitude in each band is computed as:

$$m_\lambda = \text{mag} + A_V \times (R + R_V \times dR/dR_V)$$

This parameterization captures the full wavelength-dependent extinction law
with only 3 stored numbers per model per band, enabling extremely fast
evaluation for any combination of `A(V)` and `R(V)`.

In [ ]:
# Demonstrate the get_seds function for applying reddening to mag_coeffs
try:
    from brutus.core import get_seds

    # --- Build synthetic mag_coeffs (Nmodels, Nbands, 3) ---
    # If a grid was loaded earlier we grab a few rows; otherwise we
    # construct a small illustrative array by hand.
    try:
        # Use the first 5 models from the loaded grid
        sample_coeffs = models[:5].copy()
        band_labels = filt_grid if grid_available else filt
        print("Using mag_coeffs from the loaded grid (first 5 models).")
    except Exception:
        # Create synthetic coefficients for 5 "models" and 5 "bands"
        Nmodels, Nbands = 5, 5
        band_labels = ['g', 'r', 'i', 'J', 'H']
        sample_coeffs = np.zeros((Nmodels, Nbands, 3))
        # Unreddened magnitudes (index 0) -- typical main-sequence values
        sample_coeffs[:, :, 0] = np.array([[18.0, 17.2, 16.8, 15.5, 15.0],
                                            [16.5, 15.8, 15.4, 14.3, 13.9],
                                            [15.0, 14.5, 14.2, 13.4, 13.1],
                                            [20.0, 19.0, 18.5, 17.0, 16.5],
                                            [22.0, 21.0, 20.5, 19.0, 18.5]])
        # R (reddening at Rv=0, index 1) -- roughly A_lambda / A_V at Rv=0
        sample_coeffs[:, :, 1] = np.array([1.20, 0.87, 0.68, 0.29, 0.18])
        # dR/dRv (index 2) -- change per unit Rv
        sample_coeffs[:, :, 2] = np.array([0.32, 0.24, 0.18, 0.08, 0.05])
        print("Created synthetic mag_coeffs for 5 models x 5 bands.")

    print(f"  mag_coeffs shape: {sample_coeffs.shape}")
    print(f"  Interpretation: (Nmodels={sample_coeffs.shape[0]}, "
          f"Nbands={sample_coeffs.shape[1]}, coeffs=[mag, R, dR/dRv])")

    # --- Call get_seds at different extinction levels ---
    seds_av0 = get_seds(sample_coeffs, av=0.0)
    seds_av1 = get_seds(sample_coeffs, av=1.0, rv=3.3)
    seds_av2 = get_seds(sample_coeffs, av=2.0, rv=3.3)

    print(f"\nUnreddened SEDs (A_V=0)  shape: {seds_av0.shape}")
    print(f"Reddened SEDs  (A_V=1)  shape: {seds_av1.shape}")
    print(f"Reddened SEDs  (A_V=2)  shape: {seds_av2.shape}")

    print(f"\nModel 0 magnitudes across bands:")
    header = "  {:>8s}".format("Band") + "    A_V=0    A_V=1    A_V=2"
    print(header)
    for j in range(min(sample_coeffs.shape[1], len(band_labels))):
        label = band_labels[j] if j < len(band_labels) else f"band_{j}"
        print(f"  {label:>8s}   {seds_av0[0, j]:7.3f}  {seds_av1[0, j]:7.3f}  "
              f"{seds_av2[0, j]:7.3f}")

    get_seds_available = True

except Exception as e:
    print(f"Could not run get_seds demo: {e}")
    get_seds_available = False

In [ ]:
# Plot how SEDs change with increasing A_V for a single model
try:
    if not get_seds_available:
        raise RuntimeError("get_seds not available from previous cell")

    # Pick the first model for illustration
    one_model = sample_coeffs[:1]  # shape (1, Nbands, 3)
    n_bands = one_model.shape[1]
    band_indices = np.arange(n_bands)

    # Use a subset of band labels for x-axis
    plot_labels = [band_labels[j] if j < len(band_labels) else f"band_{j}"
                   for j in range(n_bands)]

    av_values = [0.0, 0.5, 1.0, 2.0]
    colors_plot = ['black', 'steelblue', 'darkorange', 'firebrick']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left panel: magnitude vs band ---
    ax = axes[0]
    for av_val, col in zip(av_values, colors_plot):
        sed = get_seds(one_model, av=av_val, rv=3.3)  # (1, Nbands)
        ax.plot(band_indices, sed[0], 'o-', color=col, lw=2,
                label=f'A(V) = {av_val}')

    ax.set_xticks(band_indices)
    ax.set_xticklabels(plot_labels, rotation=45, ha='right')
    ax.set_ylabel('Apparent Magnitude')
    ax.set_title('SED vs Extinction (Model 0)')
    ax.invert_yaxis()  # brighter = lower magnitude
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Right panel: magnitude shift relative to unreddened ---
    ax = axes[1]
    sed_ref = get_seds(one_model, av=0.0)
    for av_val, col in zip(av_values[1:], colors_plot[1:]):
        sed = get_seds(one_model, av=av_val, rv=3.3)
        delta = sed[0] - sed_ref[0]  # extinction in magnitudes
        ax.plot(band_indices, delta, 's-', color=col, lw=2,
                label=f'A(V) = {av_val}')

    ax.set_xticks(band_indices)
    ax.set_xticklabels(plot_labels, rotation=45, ha='right')
    ax.set_ylabel('Magnitude Shift  (reddened - unreddened)')
    ax.set_title('Extinction per Band (Model 0)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Effect of Dust Extinction on a Single Model SED',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'get_seds_av_comparison')
    plt.show()

    print("Plot shows how increasing A(V) dims shorter-wavelength bands more.")

except Exception as e:
    print(f"Could not create SED comparison plot: {e}")

In [ ]:
# Reddening vector demo: use return_rvec=True to inspect the direction of reddening
try:
    if not get_seds_available:
        raise RuntimeError("get_seds not available from previous cell")

    # Compute SEDs *and* reddening vectors for the first model
    one_model = sample_coeffs[:1]  # (1, Nbands, 3)
    n_bands = one_model.shape[1]
    band_indices = np.arange(n_bands)
    plot_labels = [band_labels[j] if j < len(band_labels) else f"band_{j}"
                   for j in range(n_bands)]

    seds_rv, rvecs_rv = get_seds(one_model, av=1.0, rv=3.3, return_rvec=True)

    print("Reddening vectors at A(V)=1.0, R(V)=3.3 for Model 0:")
    print(f"  rvec shape: {rvecs_rv.shape}  (Nmodels, Nbands)")
    for j in range(min(n_bands, len(plot_labels))):
        print(f"  {plot_labels[j]:>8s}:  rvec = {rvecs_rv[0, j]:.4f}")

    # --- Also retrieve the differential reddening vector ---
    seds_full, rvecs_full, drvecs_full = get_seds(
        one_model, av=1.0, rv=3.3, return_rvec=True, return_drvec=True
    )

    print(f"\nDifferential reddening vectors (dR/dRv) for Model 0:")
    for j in range(min(n_bands, len(plot_labels))):
        print(f"  {plot_labels[j]:>8s}:  drvec = {drvecs_full[0, j]:.4f}")

    # ---- Plot ----
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: reddening vector per band
    ax = axes[0]
    ax.bar(band_indices, rvecs_rv[0], color='crimson', alpha=0.7)
    ax.set_xticks(band_indices)
    ax.set_xticklabels(plot_labels, rotation=45, ha='right')
    ax.set_ylabel('Reddening Vector  R + Rv * dR/dRv')
    ax.set_title('Reddening Vector per Band')
    ax.grid(True, alpha=0.3, axis='y')

    # Panel 2: reddening direction in a 2-band colour-magnitude diagram
    # Pick the first two bands to make a colour-magnitude diagram
    ax = axes[1]
    av_sweep = np.linspace(0, 3.0, 30)
    mags_band0, mags_band1 = [], []
    for av_val in av_sweep:
        s = get_seds(one_model, av=av_val, rv=3.3)
        mags_band0.append(s[0, 0])
        mags_band1.append(s[0, 1] if n_bands > 1 else s[0, 0])

    mags_band0 = np.array(mags_band0)
    mags_band1 = np.array(mags_band1)
    colour = mags_band0 - mags_band1

    scatter = ax.scatter(colour, mags_band0, c=av_sweep, cmap='hot_r',
                         s=40, edgecolors='k', linewidths=0.5, zorder=3)
    ax.plot(colour, mags_band0, '-', color='gray', lw=1, alpha=0.6, zorder=2)
    cb = plt.colorbar(scatter, ax=ax)
    cb.set_label('A(V)')
    ax.set_xlabel(f'{plot_labels[0]} - {plot_labels[min(1, n_bands-1)]}')
    ax.set_ylabel(f'{plot_labels[0]}')
    ax.invert_yaxis()
    ax.set_title('Reddening Track (CMD)')
    ax.grid(True, alpha=0.3)

    # Panel 3: compare rvec at different Rv values
    ax = axes[2]
    rv_values = [2.5, 3.1, 3.3, 4.0, 5.0]
    cmap = plt.cm.viridis
    for i, rv_val in enumerate(rv_values):
        _, rvec_tmp = get_seds(one_model, av=1.0, rv=rv_val, return_rvec=True)
        ax.plot(band_indices, rvec_tmp[0], 'o-', lw=2, color=cmap(i / len(rv_values)),
                label=f'R(V) = {rv_val}')

    ax.set_xticks(band_indices)
    ax.set_xticklabels(plot_labels, rotation=45, ha='right')
    ax.set_ylabel('Reddening Vector')
    ax.set_title('Reddening Vector vs R(V)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Reddening Vectors from get_seds(return_rvec=True)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'get_seds_reddening_vectors')
    plt.show()

    print("\nThe reddening vector defines the direction a star moves in")
    print("magnitude space as dust extinction increases.")

except Exception as e:
    print(f"Could not create reddening vector demo: {e}")

## Summary and Key Takeaways

This tutorial has covered model grids and performance optimization in brutus:

### Key Classes

1. **GridGenerator**: Creates pre-computed model grids
   - Samples parameter space systematically
   - Computes reddening coefficients (`mag`, `R`, `dR/dRv` per band)
   - Saves to HDF5 format

2. **StarGrid**: Interactive SED queries via grid interpolation
   - Multi-linear interpolation for any point in parameter space
   - Supports all extinction and distance effects

3. **`get_seds`**: Batch reddening application
   - Applies `m = mag + Av * (R + Rv * dR/dRv)` to pre-computed coefficients
   - Orders of magnitude faster than per-model neural network evaluation
   - Used internally by `BruteForce` for fitting

### Performance Insights

- **Speed**: Pre-computed grids avoid per-model neural network calls during fitting
- **Trade-offs**: Finer EEP spacing improves interpolation accuracy but increases storage
- **Reddening**: 3-coefficient parameterization enables fast evaluation at any `(Av, Rv)`
- **Scalability**: Grids enable fitting millions of stars with `BruteForce`

### Recommended Workflows

1. **Exploration**: Use `StarEvolTrack` for flexibility at individual points
2. **Production**: Generate custom grid for your parameter space with `GridGenerator`
3. **Large surveys**: Use pre-computed grids (MIST, Bayestar) with `BruteForce`

### Next Steps

- **Tutorial 4**: Galactic Priors and Population Synthesis
- **Tutorial 5**: Fitting Individual Sources with BruteForce
- **Tutorial 6**: Cluster Analysis and Population Fitting
- **Tutorial 7**: 3D Dust Mapping

In [ ]:
print("Tutorial 3 Complete!")
print("="*60)
print("\nGenerated files and plots:")
for file in sorted(plots_dir.glob('*')):
    size = file.stat().st_size / 1024**2 if file.suffix == '.h5' else file.stat().st_size / 1024
    unit = 'MB' if file.suffix == '.h5' else 'KB'
    print(f"  - {file.name} ({size:.1f} {unit})")